# Titel der Analyse

Untertitel

Dein Name

## LLM-Setup

Konfiguriere deinen AI-Helfer einmalig vor Beginn deiner Analyse. Führe die Code-Zellen in diesem Abschnitt bei jeder Verwendung dieses Notebooks einmalig aus.

Wenn etwas schief geht, kannst du die folgenden Zellen nochmals ausführen, um in den Startzustand zurückzukehren.

### Ollama Rechenwerk

In [ ]:
import ollama

Entscheide dich, ob du **entweder** Ollama-Cloud **oder** einen lokalen Ollama-Server verwenden willst. Wo läuft dein Modell?

#### Cloud

Dein Computer ist zu langsam oder hat nicht genügend RAM (16GB+)? Dann verwende Ollama Cloud. Informationen zu Ollama-Cloud findest du unter [docs.ollama.com/cloud](https://docs.ollama.com/cloud#cloud-models)

In [ ]:
OLLAMA_API_KEY="9d3aa5f6cdea4c549ecb32a9ed7e0446.8in8Dd5NcFYhzLObRIb_uz0g"

Deinen eigenen privaten API-Schlüssel erstellst du nach dem Login unter [ollama.com/settings/keys](https://ollama.com/settings/keys).

In [ ]:
client = ollama.Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

### Hilfsfunktionen

Für die einfachere Handhabung nutzen wir die zwei selbstgeschriebenen Funktionen `ask()` und `chat()`. Unser Code im Notebook ist dann übersichtlicher.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import json
import os

def load_conversation_log(filename = "llm_conversation_log.json"):
    """Liest ein .json-Datei, welche eine Aufzeichnung einer Unterhaltung enthält, ein."""
    global messages
    if os.path.exists(filename):
        with open(filename, 'r') as file:
            content = json.loads(file.read())
        if content[0].get("role") == "system":
            print("Loading", filename)
            messages = content

def save_conversation_log(filename = "llm_conversation_log.json"):
    """Schreibt die aktuelle Aufzeichnung der Unterhaltung in eine .json-Datei."""
    with open( filename , "w" ) as file:
        json.dump( messages , file )
    
def ask(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell - ohne Historie"""
    global client
    global model_name
    response = client.generate(model=model_name, prompt=prompt)
    display(Markdown(response.response))

def remove_message(msg_list, n=1):
    """Entfernt die ältesten n einträge aus einer Unterhaltung"""
    #print("Cleanup - removing {} oldest message from chat history".format(n))
    return msg_list[0:1] + msg_list[(n+1):]
    
def chat(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell und Historie"""
    global client
    global model_name
    global messages
    global max_chat_history
    
    messages.append({"role": "user", "content": prompt})

    if max_chat_history != 0:
        message_count = len(messages) - 1
        
        # remove exess chat history, if we have fixed size max_chat_history
        excess_messages = message_count - max_chat_history
        if max_chat_history > 0 and excess_messages > 0:
            messages = remove_message(messages, excess_messages)

        # if context window is (still) to long, we always remove old chat entries
        for _ in range(message_count + 1):
            try:
                response = client.chat(model=model_name, messages=messages)
            except ollama.ResponseError as e:
                if len(messages) > 1:
                    messages = remove_message(messages, 1)
                    continue
                raise
            except Exception as e:
                raise
    else:
        # no context lenght limit - expect your client to die eventually
        response = client.chat(model=model_name, messages=messages)
        
    messages.append({"role": "assistant", "content": response.message.content})
    save_conversation_log()
    display(Markdown(response.message.content))

def show_models_from_(output):
    """Gibt die Modelle von client.list() als Pandas Dataframe zurück"""
    df = pd.DataFrame([
        {
            "Model": entry.get("model", ""),
            "Size [GB]": round(int(entry.get("size", 0)) / 1024**3, 1)
        }
        for entry in output.get("models", [])
    ])

    if not df.empty:
        df = df.sort_values(by="Model", key=lambda s: s.str.lower())

    df = df.reset_index(drop=True)
    return df

Wichtig: Die globalen Variablen `model_name` und `messages` und `max_chat_history` müssen vor der Verwendung definiert werden. Dazu alle Code-Zellen bis **Modelltest** einmalig nacheinander ausführen. 

### Chat-History


Die Funktion `chat()` verwendet auch die bisherige Konversation als Eingabe - so "erinnert" sich ein Modell, was alles schon besprochen wurde. Die maximal Länge der Eingabe ist von Modell zu Modell verschieden aber immer begrenzt - genau wie die maximale Länge der Ausgabe eines Modells. Mit der globalen Variable `max_chat_history` kannst du einstellen, wie viele der vorhergehenden Fragen und Antworten im `chat()` verwendet werden.

|Wert|Bedeutung|
|--|--|
|-1|Model Limit - das heisst, dass dein Chat-Client soviel History wie möglich verwendet. Wenn du einen langsamen Computer hast, ist diese Einstellung keine gute Idee. Wenn du Ollama-Cloud verwendest oder einen sehr schnellen Computer hast, dann ist ggf. -1 eine gute Wahl|
|0|ohne Limit - das heisst, dass dein Chat-Client irgendwann abstützen wird - nur zu empfehlen zum Testen des "maximum context window" - also der maximalen Eingabelänge eines Modells.|
|3,4,5,...|festes Limit - Wenn du bspw. 3 wählst, dann "erinnert" sich dein Chatbot maximal an die drei vorhergehenden Fragen und Antworten. Wenn Ollama lokal läuft und/oder dein Computer langsam ist, solltest du eine kleine Zahl wie bspw. `5` wählen. Versuch macht kluch!|

In [ ]:
max_chat_history = 7

### Modellauswahl

Modellübersicht: https://ollama.com/search

Welche Modelle gibt es auf deinem Ollama-Server oder bei Ollama-Cloud?

In [ ]:
show_models_from_(client.list())

Wenn Ollama lokal läuft, achte auf die Grösse des Modells (in GB) im Verhältnis zum verfügbaren Arbeitsspeicher. Grösser heisst i.d.R. auch langsamer - aber nicht zwangsläufig auch immer besser.

Mit welchem Modell willst du arbeiten?

In [ ]:
model_name = 'gemma3:4b' # Starte mit diesem, wenn du Ollama-Cloud verwendest

#model_name = 'gemma3:270m' # Starte mit diesem, wenn du Ollama lokal laufen lässt
#model_name = 'gemma3:1b'

# weitere Beispiele
#model_name = 'deepseek-r1:1.5b'
#model_name = 'codegemma:2b'

Wenn du **nicht** Ollama-Cloud verwendest, installierst du neue Modelle auf dem Ollama-Server, mit dem du gerade verbunden bist, so: (entferne in dem Fall die #)

In [ ]:
#client.pull(model_name)

oder so:

In [ ]:
#%%bash
#ollama pull gemma3:270m

### Priming

Entweder starten wir ganz frisch **ODER** wir setzen eine Unterhaltung fort.

#### Frischer Start

Wir speichern die gesamte Unterhaltung in der globalen Variablen `messages`. Am Anfang können wir das Modell noch `primen`.

In [ ]:
priming = """
You are the best Python coder in the world.
You know Pandas and all of Pandas functionality inside out.
You use directly Pandas data frames for plotting and do NOT use matplotlib or seaborn explicitly.
You code like a student in 10th grade and prefer simple solutions.
You give short and informative answers.
"""

In [ ]:
messages = [
    {"role": "system", "content": priming}
]

#### Unterhaltung fortsetzen

Falls vorhanden, wird die Datei `llm_conversation_log.json` in die globale Variable `messages` eingelesen.

In [ ]:
load_conversation_log()

### Modelltest

Wir testen, ob alle Variablen und Funktionen vorhanden sind - falls nicht, hast du vermutlich vergessen, eine der vorhergehenden Code-Zellen auszuführen.

In [ ]:
MUST_HAVE_OBJECTS = ('ollama', 'client', 'chat', 'ask', 'max_chat_history', 'model_name', 'priming', 'messages')

llm_setup_errors = 0
for one_object in MUST_HAVE_OBJECTS:
    if one_object not in locals():
        llm_setup_errors += 1
        print("ERROR:", one_object, "nicht gefunden - bitte die zugehörige, vorhergehende Code-Zelle einmalig ausführen!")
        
assert llm_setup_errors == 0

Ausserdem testen wir, ob du eine Modell ausgewählt hast, was auch auf dem Ollama-Server vorhanden ist.

In [ ]:
assert model_name in list(show_models_from_(client.list())["Model"])

Okay, lass uns das aufschlüsseln! Hier ist der Prozess mit Beispielen:

    Daten laden: df = pd.read_csv('deine_datei.csv')
    Daten verstehen: df.head(), df.info(), df.describe() (wie vorher)
    Daten vorbereiten: Bereinigen (Fehlerhafte Werte, fehlende Daten)
    Analyse: Verwende .groupby(), .pivot_table(), .corr() wie du kennst.
    Visualisierung: Verwende df.plot() für einfache Plots.

Beispiele für Visualisierungen:

    Barplot: df['Spalte'].value_counts().plot(kind='bar') (zeigt die Häufigkeit von Werten in einer Spalte)
    Lineplot: df.plot(x='Datum', y='Wert', kind='line')
    Scatterplot: df.plot(x='Spalte1', y='Spalte2', kind='scatter')
    Boxplot: df.boxplot(column='Spalte')
    Violinplot: df.plot(kind='violin', column='Spalte')
    Heatmap (Korrelation): df.corr().plot(kind='heatmap')

Wichtig: kind ist der Parameter, der bestimmt, welchen Plot du erzeugst.

Welche Art von Daten hast du und was möchtest du visualisieren? Dann kann ich dir spezifischere Hinweise geben.

## Forschungsfragen

1. ...
2. ...
3. ...

## Daten einlesen

In [ ]:
chat("""
Wie liest man eine komma-separierte Datei in Python mit Pandas ein, 
wenn das Trennzeichen ein Semikolon ist? Was ändert sich, wenn das Trennzeichen ein Komma ist?
""")

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('KSZO-solar-data.csv', sep=",")
df

In [ ]:
df.info()

In [ ]:
df.describe()

## Daten vorverarbeiten

Zum Rechnen müssen wir die Stings in DateTime-Objekte umwandeln.

In [ ]:
chat("""
Convert this into datetime

df['time_local']

time_local
0 	2023-12-28T15:45:00
1 	2023-12-28T16:00:00
2 	2023-12-28T16:15:00
3 	2023-12-28T16:30:00
4 	2023-12-28T16:45:00
""")

In [ ]:
df['time_local'] = pd.to_datetime(df['time_local'])
df['time_utc'] = pd.to_datetime(df['time_utc'])

Wir wollen später aggregieren, z.B. nach Jahr und Monat und Stunde.

In [ ]:
chat("""
Wie berechnet man

- Kalenderwoche
- Stunde
- Wochentag
- Monat
- Quartal
- Jahr

aus einem Zeitstempel wie '2023-12-28 16:45:00'?
""")

In [ ]:
df.loc[:,'hour']  = df['time_local'].dt.hour
df.loc[:,'month'] = df['time_local'].dt.month

In [ ]:
df.info()

In [ ]:
df

## Daten analysieren

## Daten visualisieren